# Liu et al. (2025) DARM: Panasonic 25°C → 0°C

This notebook runs the manifest-controlled experiment. Target Cycle 1 is unlabelled during adaptation; target SOC is used only for final evaluation.

Start with `RUN_MODE = "smoke"`. Switch to `"formal"` only after the manifest and GPU checks pass.

In [9]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/zhangcastle/battery-soc-transfer-benchmark.git"
REPO_ROOT = Path("/content/battery-soc-transfer-benchmark")
DATA_ROOT = Path("/content/drive/MyDrive/battery_soc_benchmark/data/raw/Panasonic_18650PF")
OUTPUT_ROOT = Path("/content/drive/MyDrive/battery_soc_benchmark/outputs/liu_2025/25c_to_0c")
RUN_MODE = "smoke"  # "smoke" or "formal"
SEED = 42

assert RUN_MODE in {"smoke", "formal"}


In [ ]:
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repository: {REPO_ROOT}")
print(f"Data:       {DATA_ROOT}")
print(f"Outputs:    {OUTPUT_ROOT}")


In [ ]:
import torch

assert torch.cuda.is_available(), "Select a Colab GPU runtime before training."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 2**30:.1f} GiB")
print("L4 memory is sufficient; formal-run duration is the main constraint.")


In [ ]:
import pandas as pd

from battery_benchmark.reproductions.liu_2025.protocol import load_experiment_manifest

MANIFEST = REPO_ROOT / "data/manifests/liu_2025/panasonic_25c_to_0c.csv"
manifest = load_experiment_manifest(MANIFEST, DATA_ROOT, verify_hashes=True)
display(manifest[["record_id", "temperature_c", "role", "profile", "label_usage"]])
display(manifest.groupby(["role", "label_usage"]).size().rename("files"))
assert manifest.loc[manifest.role == "target_adapt", "label_usage"].tolist() == ["forbidden"]


In [ ]:
script = REPO_ROOT / "scripts/reproductions/liu_2025/run_reproduction.py"
run_output = OUTPUT_ROOT / RUN_MODE
command = [
    sys.executable,
    str(script),
    "--manifest", str(MANIFEST),
    "--data-root", str(DATA_ROOT),
    "--output", str(run_output),
    "--device", "cuda",
    "--seed", str(SEED),
]
if RUN_MODE == "smoke":
    command += ["--smoke", "--max-windows", "2048"]
else:
    command += ["--resume"]

print(" ".join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

result = json.loads((run_output / "results.json").read_text())
rows = []
for method, key in [("Source only", "target_source_only"), ("DARM", "target_darm")]:
    for profile, metrics in result[key].items():
        rows.append({"method": method, "profile": profile, **metrics})
metrics = pd.DataFrame(rows)
display(metrics)
metrics.pivot(index="profile", columns="method", values="rmse_percent").plot.bar(
    ylabel="RMSE (%)", rot=0, figsize=(9, 4)
)
plt.title(f"25°C → 0°C ({RUN_MODE})")
plt.tight_layout()
plt.show()


## Interpretation

- Smoke metrics only prove that the complete real-data path runs; they are not comparable with the paper.
- Formal results are comparable only with the reconstruction assumptions recorded in `results.json`.
- `source_last.pt` and `adapt_last.pt` support resuming interrupted formal runs.